Voici le pipeline professionnel complet utilisé par les data scientists en banque pour un projet de **scorecard de crédit**, basé sur les standards de l'industrie (CRISP-DM, méthodologie de scorecard type Siddiqi, et pratiques bancaires). Je présente la structure et l'objectif de chaque étape ; les détails pourront être approfondis ensuite.

---

## Pipeline de développement d'une scorecard de crédit (banque)

### Phase 0 — Cadrage business & conception du modèle (Business Understanding)
- **Objectif** : aligner le modèle sur la stratégie business et définir le cadre théorique.
- Définition de l'objectif (taux d'approbation, minimisation des défauts, rendement ajusté au risque).
- Choix du type de scorecard (application / comportemental / recouvrement).
- Définition de la population cible et de l'unité d'analyse (client / produit).
- Rédaction du design du modèle : population, fenêtres d'observation et de performance, hypothèses, sources de données.

### Phase 1 — Collecte & extraction des données (Data Understanding)
- **Objectif** : constituer un jeu de données historique représentatif et de qualité.
- Extraction des données d'application, bureau de crédit, données internes et outcomes.
- Vérification de la taille d'échantillon (minimum ~2 000 cas "mauvais").
- Audit qualité (manquants, incohérences, doublons, biais de sélection).

### Phase 2 — Définition de la variable cible (Target Definition)
- **Objectif** : formaliser ce qu'est un "bon" et un "mauvais" client.
- Définition opérationnelle du défaut (ex. 90+ jours de retard dans 12–24 mois).
- Fenêtre d'observation (caractéristiques connues à la date de scoring) et fenêtre de performance (période de réalisation du défaut).
- Règles d'exclusion (fraudes, clients indéterminés, cas limites) et hypothèses documentées.

### Phase 3 — Analyse exploratoire (EDA)
- **Objectif** : comprendre les données et les relations avec le défaut.
- Statistiques descriptives, distributions, détection d'outliers.
- Analyse de la relation variables ↔ outcome (tendances de risque, monotonicité).
- Identification des variables candidates et premières hypothèses.

### Phase 4 — Préparation des données (Data Preparation)
- **Objectif** : nettoyer et structurer les données pour la modélisation.
- Traitement des valeurs manquantes, anomalies, encodage.
- Création des variables dérivées et agrégations.
- Découpage train / validation / test (temporel de préférence).

### Phase 5 — Ingénierie des variables (Feature Engineering)
- **Objectif** : créer des prédicteurs pertinents et robustes.
- Construction de ratios, indicateurs comportementaux, variables bureau.
- Réduction de la redondance (corrélations, VIF).
- Sélection initiale par IV (Information Value) et pertinence métier.

### Phase 6 — Binning optimal & Weight of Evidence (WoE)
- **Objectif** : transformer les variables en catégories monotones et calculer le WoE.
- Binning optimal (catégories montrant une tendance de risque monotone).
- Calcul du WoE : ln(% Bons / % Mauvais).
- Évaluation de l'IV : variables avec IV > 0,1 généralement prédictives.
- Vérification de la stabilité et de la cohérence métier des bins.

### Phase 7 — Construction du modèle (Model Building)
- **Objectif** : estimer la probabilité de défaut par régression logistique sur variables WoE.
- Régression logistique (interprétable, auditable, réglementaire).
- Sélection finale des variables (significativité statistique + stabilité + métier).
- Vérification de la multicolinéarité et des coefficients.

### Phase 8 — Création de la scorecard (Scorecard Scaling)
- **Objectif** : convertir le modèle en points interprétables.
- Conversion des coefficients en points : Points = (β × WoE × Factor) + Offset.
- Paramétrage du scaling (PDO, typiquement 20–50 points pour doubler les odds ; Factor = PDO/ln(2)).
- Définition du score final et de la plage de scores.

### Phase 9 — Validation statistique
- **Objectif** : mesurer la performance prédictive et la stabilité.
- Tests statistiques : KS > 40, AUC > 0,70, Gini > 0,40.
- Tests de stabilité : PSI < 0,1 sur populations stables.
- Rank ordering, analyse des overrides, courbes de calibration.

### Phase 10 — Validation métier & stratégie de décision
- **Objectif** : traduire la performance statistique en impact business.
- Définition des cut-offs (seuils d'approbation / revue / refus).
- Analyse d'impact (taux d'approbation vs taux de défaut).
- Stress testing économique et scénarios défavorables.
- Validation par les équipes métier et risque.

### Phase 11 — Documentation & gouvernance réglementaire
- **Objectif** : produire la traçabilité et la conformité réglementaire.
- Documentation complète du modèle (méthodologie, hypothèses, données, résultats).
- Audit trail pour la supervision (Bâle, régulateurs locaux).
- Revue par le comité modèle / risk committee.

### Phase 12 — Déploiement (Deployment)
- **Objectif** : mettre le modèle en production et l'intégrer au système de décision.
- Implémentation dans le moteur de décision / système de scoring.
- Intégration aux processus d'octroi et aux canaux.
- Tests d'acceptation utilisateur (UAT) et mise en production.

### Phase 13 — Monitoring & maintenance (Post-déploiement)
- **Objectif** : surveiller la dégradation et déclencher la recalibration.
- Revues mensuelles de performance.
- Évaluations trimestrielles de stabilité (PSI, drift).
- Validation annuelle du modèle.
- Stress testing économique périodique.
- Plan de recalibration / redevelopment selon les seuils d'alerte.

---

### Cadre méthodologique sous-jacent
- **CRISP-DM** (6 phases : compréhension business, données, préparation, modélisation, évaluation, déploiement) comme squelette.
- **Cadre théorique** : variable dépendante (statut crédit), variables indépendantes, hypothèses testables.
- **Design du modèle** : unité d'analyse, population, définitions opérationnelles, fenêtres temporelles, sources de données.

Souhaites-tu que j'approfondisse l'une de ces phases (ex. le binning/WoE, le scaling des points, la validation, ou le monitoring) ?

In [1]:
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import pandas as pd

In [24]:
# Import des donnees

# Dans le notebook Phase 7
X_train_woe = pd.read_parquet("../data/processed/X_train_woe.parquet")
X_test_woe = pd.read_parquet("../data/processed/X_test_woe.parquet")
X_test_woe = pd.read_parquet("../data/processed/X_test_woe.parquet")

y_val = pd.read_parquet("../data/processed/y_val.parquet").squeeze()
y_train = pd.read_parquet("../data/processed/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/processed/y_test.parquet").squeeze()

In [25]:
X_train_woe

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone
index,,,,,,,,,,,,,,,,,,
10,-0.415165,0.251314,-0.082842,-0.180353,0.128712,-0.301787,-0.579034,-0.100083,-0.246088,-0.071459,-0.034804,-0.587015,0.174622,-0.468861,-0.035545,0.039556,0.005666,-0.044150
82,1.184134,-0.176130,-0.082842,-0.241162,0.239754,-0.098061,0.074691,-0.100083,-0.246088,0.036368,0.051449,-0.587015,0.174622,-0.468861,-0.035545,0.104261,0.005666,-0.044150
827,1.184134,-0.176130,-1.090920,-0.241162,-0.311780,-0.301787,0.074691,0.211784,0.163420,-0.010050,-0.034804,0.264053,-0.308301,0.214468,0.060961,0.039556,-0.029853,-0.044150
410,-0.415165,-0.176130,-0.082842,0.384846,0.239754,-0.301787,0.194156,-0.149489,-0.246088,0.036368,-0.034804,-0.154151,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521
48,1.184134,0.499776,0.699082,-0.180353,-0.311780,-0.301787,0.074691,0.292136,0.163420,0.036368,0.051449,0.264053,0.174622,0.214468,0.060961,0.104261,0.005666,-0.044150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,1.184134,-0.474623,-0.082842,-0.111591,0.251314,-0.301787,0.074691,-0.149489,-0.246088,-0.010050,-0.034804,-0.117337,0.174622,0.214468,-0.035545,-0.273497,0.005666,0.066521
759,-0.763916,0.251314,0.699082,-0.180353,-0.310993,-0.301787,0.194156,-0.149489,0.163420,-0.010050,0.051449,0.264053,0.174622,0.214468,0.060961,0.039556,0.005666,-0.044150
298,1.184134,-0.176130,-0.082842,-0.111591,0.239754,-0.301787,0.074691,-0.100083,0.163420,0.036368,0.471278,0.264053,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521


In [19]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [26]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def check_vif(X, threshold=10):
    """Compute VIF and return variables to drop."""
    vif_df = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }).sort_values("VIF", ascending=False)
    to_drop = vif_df[vif_df["VIF"] > threshold]["feature"].tolist()
    return vif_df, to_drop

vif_df, to_drop = check_vif(X_train_woe)
print(vif_df)
print("Variables à éliminer (VIF > 10) :", to_drop)

                    feature       VIF
10                 property  1.420950
13                  housing  1.389527
2            credit_history  1.285674
15                      job  1.267162
17                telephone  1.247017
14         existing_credits  1.193840
8       personal_status_sex  1.185561
9           residence_since  1.184039
11                      age  1.179976
1                  duration  1.173917
0           checking_status  1.153632
6                employment  1.152287
16           num_dependents  1.132850
5                   savings  1.097013
12  other_installment_plans  1.095452
7          installment_rate  1.069152
4             credit_amount  1.065643
3                   purpose  1.045249
Variables à éliminer (VIF > 10) : []


In [27]:
import statsmodels.api as sm

def fit_logistic(X, y):
    """Fit a logistic regression with statsmodels (p-values)."""
    X_const = sm.add_constant(X)
    model = sm.Logit(y, X_const).fit(disp=0)
    return model

model = fit_logistic(X_train_woe, y_train)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                    res   No. Observations:                  700
Model:                          Logit   Df Residuals:                      681
Method:                           MLE   Df Model:                           18
Date:                Wed, 18 Nov 2026   Pseudo R-squ.:                  0.2769
Time:                        15:18:04   Log-Likelihood:                -309.19
converged:                       True   LL-Null:                       -427.61
Covariance Type:            nonrobust   LLR p-value:                 3.844e-40
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.8594      0.103     -8.309      0.000      -1.062      -0.657
checking_status            -0.7787      0.135     -5.781      0.000      -1.043      -0.515


In [22]:
X_train_woe

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,other_debtors,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker
index,,,,,,,,,,,,,,,,,,,,
10,-0.415165,0.251314,-0.082842,-0.180353,0.128712,-0.301787,-0.579034,-0.100083,-0.246088,2.220446e-16,-0.071459,-0.034804,-0.587015,0.174622,-0.468861,-0.035545,0.039556,0.005666,-0.044150,2.220446e-16
82,1.184134,-0.176130,-0.082842,-0.241162,0.239754,-0.098061,0.074691,-0.100083,-0.246088,2.220446e-16,0.036368,0.051449,-0.587015,0.174622,-0.468861,-0.035545,0.104261,0.005666,-0.044150,2.220446e-16
827,1.184134,-0.176130,-1.090920,-0.241162,-0.311780,-0.301787,0.074691,0.211784,0.163420,2.220446e-16,-0.010050,-0.034804,0.264053,-0.308301,0.214468,0.060961,0.039556,-0.029853,-0.044150,2.220446e-16
410,-0.415165,-0.176130,-0.082842,0.384846,0.239754,-0.301787,0.194156,-0.149489,-0.246088,2.220446e-16,0.036368,-0.034804,-0.154151,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521,2.220446e-16
48,1.184134,0.499776,0.699082,-0.180353,-0.311780,-0.301787,0.074691,0.292136,0.163420,2.220446e-16,0.036368,0.051449,0.264053,0.174622,0.214468,0.060961,0.104261,0.005666,-0.044150,2.220446e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,1.184134,-0.474623,-0.082842,-0.111591,0.251314,-0.301787,0.074691,-0.149489,-0.246088,2.220446e-16,-0.010050,-0.034804,-0.117337,0.174622,0.214468,-0.035545,-0.273497,0.005666,0.066521,2.220446e-16
759,-0.763916,0.251314,0.699082,-0.180353,-0.310993,-0.301787,0.194156,-0.149489,0.163420,2.220446e-16,-0.010050,0.051449,0.264053,0.174622,0.214468,0.060961,0.039556,0.005666,-0.044150,2.220446e-16
298,1.184134,-0.176130,-0.082842,-0.111591,0.239754,-0.301787,0.074691,-0.100083,0.163420,2.220446e-16,0.036368,0.471278,0.264053,0.174622,0.214468,-0.035545,0.039556,0.005666,0.066521,2.220446e-16


In [28]:
import pandas as pd

# Absolute correlation matrix
corr_matrix = X_train_woe.corr().abs()

# Mask to keep only the upper triangle (avoid duplicates and diagonal)
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find pairs with near-perfect correlation (e.g. > 0.99 or == 1.0)
threshold = 0.99
collinear_pairs = []

for col in upper_tri.columns:
  high_corr = upper_tri.index[upper_tri[col] >= threshold].tolist()
  for row in high_corr:
    collinear_pairs.append({
        "variable_1": row,
        "variable_2": col,
        "correlation": round(corr_matrix.loc[row, col], 4),
    })

df_collinear = pd.DataFrame(collinear_pairs)
print("=== Paires quasi-colinéaires ===")
print(
    df_collinear.to_string(index=False)
    if not df_collinear.empty
    else "Aucune paire au-dessus du seuil."
)

=== Paires quasi-colinéaires ===
Aucune paire au-dessus du seuil.


7.4/ Ajutsement du moelede (regresiion logistique)

In [30]:
import statsmodels.api as sm

def fit_logistic(X, y):
    """Fit a logistic regression with statsmodels (p-values)."""
    X_const = sm.add_constant(X)
    model = sm.Logit(y, X_const).fit(disp=0)
    return model

model = fit_logistic(X_train_woe, y_train)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                    res   No. Observations:                  700
Model:                          Logit   Df Residuals:                      681
Method:                           MLE   Df Model:                           18
Date:                Wed, 18 Nov 2026   Pseudo R-squ.:                  0.2769
Time:                        15:28:05   Log-Likelihood:                -309.19
converged:                       True   LL-Null:                       -427.61
Covariance Type:            nonrobust   LLR p-value:                 3.844e-40
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.8594      0.103     -8.309      0.000      -1.062      -0.657
checking_status            -0.7787      0.135     -5.781      0.000      -1.043      -0.515


In [ ]:
# With SKLEARN (code to use in production)

from sklearn.linear_model import LogisticRegression

def fit_logistic_sklearn(X, y, C=1.0):
    """Fit a logistic regression with sklearn."""
    model = LogisticRegression(C=C, solver="lbfgs", max_iter=1000)
    model.fit(X, y)
    return model

model_sk = fit_logistic_sklearn(X_train_woe, y_train)

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status_sex,other_installment_plans
index,,,,,,,,,,
10,-0.415165,0.251314,-0.082842,-0.180353,0.128712,-0.301787,-0.579034,-0.100083,-0.246088,0.174622
82,1.184134,-0.176130,-0.082842,-0.241162,0.239754,-0.098061,0.074691,-0.100083,-0.246088,0.174622
827,1.184134,-0.176130,-1.090920,-0.241162,-0.311780,-0.301787,0.074691,0.211784,0.163420,-0.308301
410,-0.415165,-0.176130,-0.082842,0.384846,0.239754,-0.301787,0.194156,-0.149489,-0.246088,0.174622
48,1.184134,0.499776,0.699082,-0.180353,-0.311780,-0.301787,0.074691,0.292136,0.163420,0.174622
...,...,...,...,...,...,...,...,...,...,...
603,1.184134,-0.474623,-0.082842,-0.111591,0.251314,-0.301787,0.074691,-0.149489,-0.246088,0.174622
759,-0.763916,0.251314,0.699082,-0.180353,-0.310993,-0.301787,0.194156,-0.149489,0.163420,0.174622
298,1.184134,-0.176130,-0.082842,-0.111591,0.239754,-0.301787,0.074691,-0.100083,0.163420,0.174622


7.5/ Backward Elimination

In [31]:
def backward_elimination(X,y,p_threshold=0.05):
    """Remove variables with p_value > threshold."""
    features= list(X.columns)
    while True :
        # Add the statsmodels constant
        X_const=sm.add_constant(X[features])
        # cree le modele de regresion et l'entraine
        model=sm.Logit(y, X_const).fit(disp=0)
        # recuperer la p_value et envever l'intercept de la liste
        p_values=model.pvalues.drop("const")
        # find the variable name with the highest p-value
        worst=p_values.idxmax()

        if p_values[worst] >  p_threshold:
            features.remove(worst)
            print(f"Éliminé : {worst} (p={p_values[worst]:.4f})")
        else:
            break
    return features, model

selected_features, final_model =backward_elimination(X_train_woe,y_train)
print("Variables retenues :", selected_features)

Éliminé : residence_since (p=0.9085)
Éliminé : job (p=0.4702)
Éliminé : existing_credits (p=0.3668)
Éliminé : property (p=0.2947)
Éliminé : num_dependents (p=0.2732)
Éliminé : age (p=0.2244)
Éliminé : telephone (p=0.1438)
Éliminé : housing (p=0.0538)
Variables retenues : ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings', 'employment', 'installment_rate', 'personal_status_sex', 'other_installment_plans']


In [44]:
# skleanr model with ONLY the selected features
final_model_sk=fit_logistic_sklearn(X_train_woe[selected_features],y_train)

In [ ]:
final_param=final_model.params

const                     -0.866498
checking_status           -0.820276
duration                  -0.961775
credit_history            -0.704936
purpose                   -1.007703
credit_amount             -0.932540
savings                   -0.914025
employment                -0.711636
installment_rate          -1.814027
personal_status_sex       -1.441575
other_installment_plans   -0.747594
dtype: float64

7.6/ Verification de coefficients

We have to verify that the coefficients are consistentnwith the business context

In [32]:
# Model coefficients
coef_df = pd.DataFrame({
    "feature": ["const"] + selected_features,
    "coefficient": final_model.params.values,
    "p_value": final_model.pvalues.values
})
print(coef_df)

# Check: signs must be consistent with business meaning
# E.g. high WoE = lower risk -> negative coefficient expected

                    feature  coefficient       p_value
0                     const    -0.866498  2.525688e-17
1           checking_status    -0.820276  5.343482e-10
2                  duration    -0.961775  2.035666e-06
3            credit_history    -0.704936  5.829988e-04
4                   purpose    -1.007703  2.591785e-04
5             credit_amount    -0.932540  1.409938e-05
6                   savings    -0.914025  6.065907e-05
7                employment    -0.711636  7.385190e-03
8          installment_rate    -1.814027  1.253080e-03
9       personal_status_sex    -1.441575  5.523042e-03
10  other_installment_plans    -0.747594  8.639264e-03


Comment : All the p_values are higher than 0.05, hence all is okay on that front

7.7/ Sauvegarde du modele

list

In [45]:
import joblib

# sauvegarder le modele pkl pour la prod
joblib.dump(final_model_sk,"../models/logistic_model.pkl")

# sauvegarder la liste des features retenues

import json
with open("../models/selected_features.json","w") as f:
    json.dump(selected_features,f)
